# 02. Generators

**Tier:** Deep. **Why you'd use this:** You need the squares of a long range of numbers, but you will only read a few of them. Building the whole list would waste memory. A generator hands out values one at a time.

**After this lesson you can:** write a generator with `yield`; measure that it stays small; explain why a second loop over the same generator is empty.
About 30 minutes. **Uses:** [Comprehensions](01-comprehensions-lesson.ipynb)


## Quick review

1. Why does `item` outside a comprehension stay unchanged?
2. Does a comprehension walk every source item even when `if` filters most of them out?

<details><summary>Answers</summary>

The comprehension has its own scope. Yes. The filter skips appends, not the walk.
</details>


## The idea

A generator function uses `yield` instead of building a list. Calling it returns a generator object. Each `next` runs until the next `yield` and pauses. `for` is `iter` plus `next` until `StopIteration`. A generator is the iterator: walking it once uses it up. A second loop sees nothing. That difference from a list is the whole point of this lesson.

<details><summary>Explain more</summary>

`yield value` hands `value` to the caller and freezes the frame. The next pull resumes after that line. Locals stay alive between pulls. When the function returns, the iterator is spent. A list comprehension allocates every result up front. A generator does not. You can stop after three pulls and never pay for the rest.
</details>


## See it work

Hand out squares one at a time, then show that a second walk is empty.


In [1]:
def squares(nums):
    # 1. Pause after each value instead of collecting them.
    for value in nums:
        yield value * value


stream = squares([1, 2, 3])
print(list(stream))
print(list(stream))


[1, 4, 9]
[]


## Check yourself

Answer these before you run the cell.

1. Which object is smaller, the list or the generator?
2. What does the second `list(gen)` print?


In [2]:
import sys

values = range(10_000)
as_list = [value * value for value in values]
as_gen = (value * value for value in values)
print(sys.getsizeof(as_list) > sys.getsizeof(as_gen))
gen = (value for value in (1, 2, 3))
print(list(gen))
print(list(gen))


True
[1, 2, 3]
[]


<details><summary>Answers</summary>

The list is larger. The second `list(gen)` is `[]`. The generator was already spent.
</details>


## Why it works this way

`for` always uses the iteration protocol: call `iter` on the subject, then `next` until `StopIteration`. A list's `iter` returns a fresh cursor each time, so a second loop starts over. A generator object is already the cursor. Its `iter` returns itself. After the last `yield`, the ending `next` raises `StopIteration` and the frame is cleared. A second loop gets that empty iterator immediately. That silent empty walk is the bug people hit when they save one generator and reuse it. Deep dive: [The iteration protocol](../../internals/07-iteration/01-protocol-lesson.ipynb).

Laziness is the design: you pay for each value when you pull it. `sys.getsizeof` on the generator stays near a small fixed size even when the logical length is huge, because the results are not stored in a container. The generator holds a frame, not the full output. Pulling every value still takes time proportional to how many you pull. The memory win disappears if you call `list` on the whole stream, because that allocates every result at once.

A generator function is ordinary Python until the first `yield`. Calling it does not run the body. It builds a generator object. Each pull runs until the next `yield` and freezes local names in that frame. That is why you can keep a running total inside the generator without exposing it. When the function returns, the frame goes away and further `next` calls raise `StopIteration`. `for` catches that exception and stops cleanly. You almost never write `except StopIteration` yourself.

Trace of `squares([1, 2])`:

| step | action | yielded |
|---|---|---|
| 1 | enter loop, `value` is 1 | 1 |
| 2 | resume, `value` is 2 | 4 |
| 3 | loop ends | `StopIteration` |

Use that table when you debug. If the second consumer sees nothing, ask whether you still hold the same generator object. Calling the function again builds a fresh cursor. Sharing one cursor between two consumers is the wrong shape unless both agree to pull from the same stream on purpose.


Generators matter most when the producer and consumer disagree about how much work is needed. A file line stream, a page of search hits, or a long `range` mapped through a formula should not force a full list just because the API is easier to write that way. Start with a generator when the natural next step is a single walk. Switch to a list when a later step needs `len`, indexing, or a second pass. Measuring with `sys.getsizeof` makes the trade-off concrete before the input grows large enough to hurt.


In [3]:
def squares(nums):
    for value in nums:
        yield value * value


stream = squares([1, 2])
iterator = iter(stream)
print(iterator is stream)
print(next(iterator))
print(next(iterator))


True
1
4


In [4]:
def squares(nums):
    for value in nums:
        yield value * value


stream = squares([1, 2])
iterator = iter(stream)
print(next(iterator))
print(next(iterator))
print(next(iterator))


1
4


StopIteration: 

## When to use it

Use a generator when the full list would be large or may not be needed. Use a list when you need length, reuse, or random access. Don't walk the same generator twice and treat an empty second loop as data. Call the generator function again for a fresh stream. Prefer a generator expression for a one-line lazy map, and a generator function when the body needs several statements or local state between yields.


## Common mistake

Saving one generator and looping twice. The second loop is silently empty, so a total looks finished when it never ran. The bug is quiet because `sum` of an empty stream is `0`, which can look like a real answer.


In [5]:
def squares(nums):
    for value in nums:
        yield value * value


stream = squares([1, 2, 3])
print(sum(stream))
print(sum(stream))


14
0


Call the function again when you need another walk. Each call builds a new generator. If two parts of the program must share one stream, pass the generator once and document that ownership. Do not hand the same object to two independent totals.


In [6]:
def squares(nums):
    for value in nums:
        yield value * value


print(sum(squares([1, 2, 3])))
print(sum(squares([1, 2, 3])))


14
14


## Try it

Change `squares` so it yields only even squares. For `[1, 2, 3, 4]`, which values come out, and why does filtering inside the generator still beat building a filtered list when the input is huge and you stop early?


## Recap

- `yield` pauses a function and hands back one value at a time.
- A generator is single-pass. A second loop is empty.
- `for` is `iter` and `next` until `StopIteration`.
- Measure with `sys.getsizeof` when you need proof that laziness saved memory.

**Next:** [practice](02-generators-practice.ipynb)
